# Kirchhoff-Love shells in `future`

The `future` module gained a rotation-free Kirchhoff-Love (KL) shell element, ported from
the legacy Fortran `UELMAT3` (`'U3'`) element. Unlike the existing 2D/3D solid path, a
shell's per-node contribution is a genuine 3x3 strain-displacement matrix `B_a`
(membrane and bending), combined as `K_ab = B_a^T . matH . B_b` -- structurally
incompatible with `ConstitutiveLaw`'s B-free pairwise-gradient identity used for solids.
`KirchhoffLoveShellLaw` is therefore a new, parallel (non-inheriting) class, and
`PatchIntegrator` gained dedicated `integrate_shell_stiffness()` / `integrate_shell_mass()`
/ `integrate_shell_surface_load()` / `assemble_shell_stiffness()` / `assemble_shell_mass()`
/ `assemble_shell_surface_load()` methods.

**Scope**: single-patch or C0-continuous multipatch shells only. C1 multipatch coupling
via bending strips (legacy `bendingstrip.f`, needed to keep folded/multi-panel shells
tangent across patch boundaries) is out of scope for this phase.

Every control point keeps 3 translational DOFs (no independent rotation field), so a
shell `Patch` needs `ControlPointManager(dim=3)` and `IGABasis1D.build(..., deriv_order=2)`
(shell curvature needs second parametric derivatives, unlike the solid path).

In [ ]:
import os
import tempfile

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve

from yeti_iga.preprocessing.igaparametrization import IGAparametrization
from yeti_iga.stiffmtrx_elemstorage import sys_linmat_lindef_static as build_stiffmatrix_legacy
from yeti_iga.future.bspline import (
    BSpline, BSplineSurface, ControlPointManager, Patch,
    GlobalDOFManager, PatchDOFManager, IGABasis1D, PatchIntegrator,
    Material, KirchhoffLoveShellLaw, PRefiner, SubdivisionRefiner,
)
from yeti_iga.future.postprocessing.vtu import write_bezier_patch_vtu
import pyvista as pv
from yeti_iga.future.postprocessing.pv_plotting import add_deformed_shell

## Part 1 -- flat plate, cross-checked against legacy

`benchs/shell_1_element_load/1_element_degree_3` is a flat 10x10 unit-square patch,
degree 3x3 (16 control points, a single Bezier element), pure B-spline (all weights 1).
Steel-like material, `E = 210 GPa`, `nu = 0.3`, thickness `0.1`. Being flat, all curvature
terms (`a11`, `a12`, `a22`) vanish -- this exercises the membrane/bending B-matrices and
`matH`, but not the 2nd-derivative NURBS quotient rule needed for curved geometry.

In [ ]:
def legacy_stiffness(filename):
    iga_model = IGAparametrization(filename=filename)
    data, row, col, _ = build_stiffmatrix_legacy(*iga_model.get_inputs4system_elemStorage())
    K = sp.coo_matrix(
        (data, (row, col)),
        shape=(iga_model.nb_dof_tot, iga_model.nb_dof_tot),
        dtype='float64').tocsc()
    return K + K.transpose()


# Path relative to this notebook's own directory (examples/future/), matching
# the convention used by 04_benchmark_future_legacy.ipynb's 'input/...' path.
fixture = os.path.join(
    '..', '..', 'benchs', 'shell_1_element_load', '1_element_degree_3')
K_legacy = legacy_stiffness(fixture)
print('legacy stiffness shape:', K_legacy.shape)

In [ ]:
# u-fastest CP order (4x4 grid), matching the fixture's *Node block.
coords = [
    (0.0, 0.0, 0.0), (3.33, 0.0, 0.0), (6.67, 0.0, 0.0), (10.0, 0.0, 0.0),
    (0.0, 3.33, 0.0), (3.33, 3.33, 0.0), (6.67, 3.33, 0.0), (10.0, 3.33, 0.0),
    (0.0, 6.67, 0.0), (3.33, 6.67, 0.0), (6.67, 6.67, 0.0), (10.0, 6.67, 0.0),
    (0.0, 10.0, 0.0), (3.33, 10.0, 0.0), (6.67, 10.0, 0.0), (10.0, 10.0, 0.0),
]
mgr = ControlPointManager(dim=3)
for c in coords:
    mgr.add_point(list(c))

kv = np.array([0., 0., 0., 0., 1., 1., 1., 1.])
su = BSpline(3, kv)
sv = BSpline(3, kv)
surf = BSplineSurface(su, sv)
mapping = list(range(16))
local_shape = [4, 4]

dofs_per_cp = [3 for _ in range(mgr.n_points)]
gdm = GlobalDOFManager(dofs_per_cp)
pdm = PatchDOFManager(3, mapping, gdm)
patch = Patch(surf, mgr, mapping, local_shape, pdm)

# deriv_order=2: shell curvature needs second parametric derivatives.
basis_u = IGABasis1D.build(su, 4, 2)
basis_v = IGABasis1D.build(sv, 4, 2)

shell_law = KirchhoffLoveShellLaw(Material(E=210e9, nu=0.3, thickness=0.1))

# No ConstitutiveLaw at construction time: shell laws are passed explicitly
# to integrate_shell_stiffness()/integrate_shell_mass() instead.
integrator = PatchIntegrator(patch, basis_u, basis_v)
K_future = integrator.integrate_shell_stiffness(shell_law)

rel_err = np.linalg.norm(K_future.toarray() - K_legacy.toarray()) / np.linalg.norm(K_future.toarray())
print('relative Frobenius error vs legacy:', rel_err)

Agreement to machine precision, as for every other future-vs-legacy comparison in this
codebase.

## Part 2 -- squareShellRoof fixture, also flat, also cross-checked

`benchs/squareShellRoof/squareShellPlate` is another single-element shell patch
(degree 1x1, 4 control points, one Bezier element). Despite the directory's name it is
**not** the classical Scordelis-Lo roof benchmark (a fixed curved cylindrical shell under
self-weight, with a well-known reference deflection) -- that benchmark does not exist yet
as a fixture in this repository. This flat fixture is instead the *starting point* for a
shape-optimization case elsewhere in the repo (`test_grad_square_shell_roof.py` optimizes
its control points' z-coordinates into an actual curved roof shape to minimize
compliance at constant volume). Here it is only used, as-is, for a second flat-plate
stiffness cross-check, with the same steel-like material as Part 1
(`E = 210 GPa`, `nu = 0.3`, thickness `0.1`).

This fixture happens to fully clamp all 4 of its control points (all 3 translational
DOFs at every corner), which triggers a legacy-side edge case unrelated to shells:
`IGAparametrization._update_dof_info()` special-cases `nb_dof_free == 0` by setting
`_ind_dof_free` to an **empty** array, whereas `sys_linmat_lindef_static` (used here to
build the *raw*, unconstrained stiffness matrix -- the same "matrices in, boundary
conditions applied separately" convention used throughout `future`) expects
`_ind_dof_free` to span every DOF. Worked around below by overriding
`_ind_dof_free`/`_nb_dof_free` directly before assembly, exactly as if no boundary
condition had been read -- boundary conditions play no role in comparing raw stiffness
matrices anyway.

In [ ]:
def legacy_stiffness_fully_clamped(filename):
    iga_model = IGAparametrization(filename=filename)
    if iga_model._nb_dof_free == 0:
        # See markdown above: sys_linmat_lindef_static needs ind_dof_free to
        # span every dof to build the raw (unconstrained) stiffness matrix.
        iga_model._ind_dof_free = np.arange(1, iga_model._nb_dof_tot + 1)
        iga_model._nb_dof_free = iga_model._nb_dof_tot
    data, row, col, _ = build_stiffmatrix_legacy(*iga_model.get_inputs4system_elemStorage())
    K = sp.coo_matrix(
        (data, (row, col)),
        shape=(iga_model.nb_dof_tot, iga_model.nb_dof_tot),
        dtype='float64').tocsc()
    return K + K.transpose()


fixture_roof = os.path.join('..', '..', 'benchs', 'squareShellRoof', 'squareShellPlate')
K_legacy_roof = legacy_stiffness_fully_clamped(fixture_roof)

# u-fastest CP order (2x2 grid), matching the fixture's *Node block.
coords_roof = [(0.0, 0.0, 0.0), (10.0, 0.0, 0.0), (0.0, 10.0, 0.0), (10.0, 10.0, 0.0)]
mgr_roof = ControlPointManager(dim=3)
for c in coords_roof:
    mgr_roof.add_point(list(c))

kv1 = np.array([0., 0., 1., 1.])
su_roof = BSpline(1, kv1)
sv_roof = BSpline(1, kv1)
surf_roof = BSplineSurface(su_roof, sv_roof)
mapping_roof = [0, 1, 2, 3]
local_shape_roof = [2, 2]

dofs_per_cp_roof = [3 for _ in range(mgr_roof.n_points)]
gdm_roof = GlobalDOFManager(dofs_per_cp_roof)
pdm_roof = PatchDOFManager(3, mapping_roof, gdm_roof)
patch_roof = Patch(surf_roof, mgr_roof, mapping_roof, local_shape_roof, pdm_roof)

basis_u_roof = IGABasis1D.build(su_roof, 2, 2)
basis_v_roof = IGABasis1D.build(sv_roof, 2, 2)

shell_law_roof = KirchhoffLoveShellLaw(Material(E=210e9, nu=0.3, thickness=0.1))
integrator_roof = PatchIntegrator(patch_roof, basis_u_roof, basis_v_roof)
K_future_roof = integrator_roof.integrate_shell_stiffness(shell_law_roof)

rel_err_roof = (np.linalg.norm(K_future_roof.toarray() - K_legacy_roof.toarray())
                / np.linalg.norm(K_future_roof.toarray()))
print('relative Frobenius error vs legacy (squareShellRoof fixture):', rel_err_roof)

Machine precision again, as expected: this fixture is flat too, so it exercises the same
membrane/bending B-matrices and `matH` as Part 1, just with a different node count and
degree.

## Part 3 -- curved NURBS shell: no legacy fixture exists

Every `'U3'` shell fixture in `benchs/` (`catenary/shellArch`, `squareShellRoof(Disp)`,
`Tbeam2cplg`) turns out to be a **flat** panel (some just tilted at an angle) -- none has
non-planar geometry or a weight != 1. So there is nothing to cross-check the curved,
rational (NURBS) shell math against.

Instead: build an *exact* NURBS quarter-cylinder shell patch (degree 2 in the hoop
direction with weight `cos(45 deg)` at the middle control point -- the standard exact
circular-arc NURBS representation -- degree 1, straight, in the axial direction), which is
genuinely curved and genuinely rational. Then check the classical **rigid body mode**
property of a consistent linear shell operator: any linearized rigid translation or
rotation applied to every control point must be exactly annihilated by the stiffness
matrix (`K @ d == 0`), because a rigid motion produces zero strain. This exercises the full
curvature/bendingB/2nd-derivative-NURBS-quotient-rule pipeline end-to-end, without
requiring a legacy reference matrix.

In [ ]:
def build_cylinder_patch(radius=2.0, length=3.0):
    w1 = np.cos(np.pi / 4)
    hoop_pts = [(radius, 0.0), (radius, radius), (0.0, radius)]
    hoop_w = [1.0, w1, 1.0]

    coords, weights = [], []
    for z in (0.0, length):
        for (x, y), w in zip(hoop_pts, hoop_w):
            coords.append((x, y, z))
            weights.append(w)

    mgr = ControlPointManager(dim=3)
    for c, w in zip(coords, weights):
        mgr.add_point(list(c), w=w)

    ku = np.array([0., 0., 0., 1., 1., 1.])
    kv = np.array([0., 0., 1., 1.])
    su = BSpline(2, ku)
    sv = BSpline(1, kv)
    surf = BSplineSurface(su, sv)
    mapping = list(range(6))
    local_shape = [3, 2]

    dofs_per_cp = [3 for _ in range(mgr.n_points)]
    gdm = GlobalDOFManager(dofs_per_cp)
    pdm = PatchDOFManager(3, mapping, gdm)
    patch = Patch(surf, mgr, mapping, local_shape, pdm)

    basis_u = IGABasis1D.build(su, 4, 2)
    basis_v = IGABasis1D.build(sv, 4, 2)

    return patch, basis_u, basis_v, np.array(coords)


shell_law_cyl = KirchhoffLoveShellLaw(Material(E=210e9, nu=0.3, thickness=0.1))
patch_cyl, basis_u_cyl, basis_v_cyl, coords_cyl = build_cylinder_patch()

K_cyl = PatchIntegrator(patch_cyl, basis_u_cyl, basis_v_cyl).integrate_shell_stiffness(shell_law_cyl).toarray()
k_norm = np.linalg.norm(K_cyl)
print('curved shell K shape:', K_cyl.shape, ' ||K|| =', k_norm)

In [ ]:
n_cp = coords_cyl.shape[0]
n_dof = 3 * n_cp
center = coords_cyl.mean(axis=0)

rigid_modes = []
for i in range(3):  # translations
    d = np.zeros(n_dof)
    d[i::3] = 1.0
    rigid_modes.append(('translation', i, d))
for i, axis in enumerate(np.eye(3)):  # infinitesimal rotations about the centroid
    d = np.zeros(n_dof)
    for a in range(n_cp):
        d[3 * a:3 * a + 3] = np.cross(axis, coords_cyl[a] - center)
    rigid_modes.append(('rotation', i, d))

for kind, axis, d in rigid_modes:
    residual = np.linalg.norm(K_cyl @ d) / k_norm
    print(f'{kind} axis {axis}: |K@d| / |K| = {residual:.3e}')

All 6 rigid body modes are annihilated to machine precision, validating the curved/NURBS
shell kinematics.

## Part 4 -- consistent mass matrix

No shell fixture in `benchs/` defines a density (all are static analyses), so there is no
legacy mass matrix to compare against either. Falling back to an analytical check on the
flat plate: for a partition-of-unity basis, `sum_a R_a(u,v) == 1` everywhere, so summing an
entire translational block of the consistent mass matrix must equal `rho * thickness * area`.

In [ ]:
rho = 7850.0
thickness = 0.1
shell_law_mass = KirchhoffLoveShellLaw(Material(E=210e9, nu=0.3, rho=rho, thickness=thickness))

M = PatchIntegrator(patch, basis_u, basis_v).integrate_shell_mass(shell_law_mass).toarray()

area = 10.0 * 10.0
expected_mass = rho * thickness * area
x_dofs = slice(0, None, 3)
total_mass = M[x_dofs, x_dofs].sum()

print('expected total mass:', expected_mass)
print('mass matrix total (x-block):', total_mass)
print('relative error:', abs(total_mass - expected_mass) / expected_mass)

## Part 5 -- full structural computation: load, clamp, solve

The `squareShellPlate` fixture's own boundary conditions and `U66` distributed pressure
load (a legacy "snow load": force per unit area along the global Z axis, scaled by the
local normal's Z component -- exactly a constant vertical vector on this flat patch,
as used in Part 2) are only interesting once the mesh has more than 4 control points: on
the raw 2x2 patch, *every* control point is a corner, so clamping all 4 corners (as the
`.inp` does) leaves nothing free to solve for -- that degeneracy is exactly what caused
Part 2's `_ind_dof_free` edge case.

Refining first -- the same way the legacy shape-optimization benchmark for this fixture
does (`OPT2-squareShellRoof.py`/`test_grad_square_shell_roof.py`: degree elevation
`[1, 1]` then 2 levels of h-refinement) -- turns this into an actual structural problem:
only the 4 *original* domain corners stay clamped, and every other control point (32 of
them, on a 6x6 grid) is free.

`integrate_shell_surface_load()` (new in this phase, alongside `integrate_shell_stiffness()`/
`integrate_shell_mass()`) integrates a **constant** force-per-unit-area vector against
`R_a` over the whole patch -- unlike `integrate_boundary_load()`, which only covers one
edge. It reproduces the legacy `U_1`/`U_2`/`U_3` Dload types exactly, and `U_0`/`U_6`
(normal pressure / "snow load") too on a flat patch like this one, where the unit normal
is itself constant; a direction-varying normal-pressure variant for curved geometry is a
natural future extension, not implemented yet.

In [ ]:
# Coarse (geometry-only) patch: same 10x10 flat square as squareShellPlate.
mgr_geo = ControlPointManager(dim=3)
for c in coords_roof:
    mgr_geo.add_point(list(c))
patch_coarse = Patch(BSplineSurface(BSpline(1, kv1), BSpline(1, kv1)), mgr_geo,
                      [0, 1, 2, 3], [2, 2])

# k-refinement: degree elevation +1 (bilinear -> biquadratic), then 2 levels
# of uniform subdivision (4 elements per direction) -- matches the legacy
# shape-optimization benchmark's mesh for this same fixture.
PRefiner(direction=0, n_elevations=1).refine(patch_coarse)
PRefiner(direction=1, n_elevations=1).refine(patch_coarse)
SubdivisionRefiner(direction=0, n_levels=2).refine(patch_coarse)
SubdivisionRefiner(direction=1, n_levels=2).refine(patch_coarse)

n_cp_r = patch_coarse.cp_manager.n_points
nu_r, nv_r = patch_coarse.local_shape
print('refined mesh: degree', patch_coarse.tensor.components[0].degree,
      patch_coarse.tensor.components[1].degree, ' shape', patch_coarse.local_shape,
      ' n_cp', n_cp_r)

# Attach a DOF manager now that refinement is done (refiners only touch geometry).
gdm_r = GlobalDOFManager([3] * n_cp_r)
pdm_r = PatchDOFManager(3, patch_coarse.global_indices, gdm_r)
patch_refined = Patch(patch_coarse.tensor, patch_coarse.cp_manager,
                       patch_coarse.global_indices, patch_coarse.local_shape, pdm_r)

su_r, sv_r = patch_refined.tensor.components
basis_u_r = IGABasis1D.build(su_r, su_r.degree + 1, 2)
basis_v_r = IGABasis1D.build(sv_r, sv_r.degree + 1, 2)

shell_law_r = KirchhoffLoveShellLaw(Material(E=210e9, nu=0.3, thickness=0.1))
integrator_r = PatchIntegrator(patch_refined, basis_u_r, basis_v_r)
K_r = integrator_r.integrate_shell_stiffness(shell_law_r)
F_r = integrator_r.integrate_shell_surface_load(np.array([0., 0., 1.]), -500.0)
print('K shape', K_r.shape, ' F shape', F_r.shape, ' total Fz', F_r[2::3].sum())

# Clamp only the 4 domain corners (all 3 translational DOFs), matching the
# *Boundary blocks in squareShellPlate.inp.
corner_local = [0, nu_r - 1, nu_r * (nv_r - 1), nu_r * nv_r - 1]
fixed_dofs = []
for cp in corner_local:
    fixed_dofs.extend(patch_refined.dof_manager.get_global_dof_indices(cp))
fixed_dofs = np.array(sorted(fixed_dofs))
free_dofs = np.setdiff1d(np.arange(K_r.shape[0]), fixed_dofs)

d_future = np.zeros(K_r.shape[0])
d_future[free_dofs] = spsolve(K_r.tocsc()[free_dofs, :][:, free_dofs], F_r[free_dofs])

print('displacement at clamped corners (must be 0):', d_future[fixed_dofs])
print('max |uz|:', np.abs(d_future[2::3]).max())

### Cross-checking the refined solve against legacy

Bridging `future`'s exact refined patch (control points, knot vectors, weights,
connectivity) to a temporary legacy `.inp`/`.NB` pair -- the same technique as
`11_future_legacy_crosscheck.ipynb`'s `write_legacy_files`, generalized here from a 2D
solid multipatch model to a single 3D shell patch -- lets the Fortran routines solve the
*exact same* mesh, guaranteeing the control-point numbering matches rather than trying to
keep two independent refinement implementations in sync by construction.
`sys_linmat_lindef_static`'s 4th return value (ignored as `_` everywhere above) is the
RHS load vector.

In [ ]:
def write_legacy_shell_files(patch, basename, E, nu, thickness, load_direction_code, load_magnitude):
    su, sv = patch.tensor.components
    p_u, p_v = su.degree, sv.degree
    nu_cp, nv_cp = patch.local_shape
    n_cp = patch.cp_manager.n_points
    nnode = (p_u + 1) * (p_v + 1)

    coords_all = np.array([patch.control_point(i) for i in range(n_cp)])
    weights_all = patch.cp_manager.weights_view() if patch.cp_manager.is_rational else np.ones(n_cp)

    elements = []
    for span in patch.spans():
        span_u, span_v = span[0], span[1]
        window_u = range(span_u - p_u, span_u + 1)
        window_v = range(span_v - p_v, span_v + 1)
        Lw = [iu + nu_cp * iv for iv in window_v for iu in window_u]  # u-fastest increasing
        global_ids_inc = [patch.global_indices[k] for k in Lw]
        IEN_row = [gid + 1 for gid in reversed(global_ids_inc)]        # 1-based, reversed
        weight_row = [weights_all[gid] for gid in reversed(global_ids_inc)]
        elements.append(((span_u + 1, span_v + 1), IEN_row, weight_row))

    nb_lines = ['*Dimension', '2', '*Number of CP by element', str(nnode),
                '*Number of patch', '1', '*Total number of element', str(len(elements)),
                '*Number of element by patch', str(len(elements)), '*Patch(1)',
                str(len(su.knot_vector)), ','.join(f'{v:.16g}' for v in su.knot_vector),
                str(len(sv.knot_vector)), ','.join(f'{v:.16g}' for v in sv.knot_vector),
                '*Jpqr', f'{p_u},{p_v}', '*Nijk']
    for eid, (Nijk, _, _) in enumerate(elements, start=1):
        nb_lines.append(f'{eid},{Nijk[0]},{Nijk[1]}')
    nb_lines.append('*Weight')
    for eid, (_, _, w_row) in enumerate(elements, start=1):
        nb_lines.append(f"{eid}," + ','.join(f'{w:.16g}' for w in w_row))
    with open(f'{basename}.NB', 'w') as fh:
        fh.write('\n'.join(nb_lines) + '\n')

    inp = ['*HEADING', '*Part, name=Piece',
           f'*USER ELEMENT, NODES={nnode}, TYPE=U3, COORDINATES=3, '
           f'INTEGRATION={nnode}, TENSOR=PSTRESS', '1,2,3', '*Node,nset=AllNode']
    for i in range(n_cp):
        x, y, z = coords_all[i]
        inp.append(f'{i + 1}, {x:.16g}, {y:.16g}, {z:.16g}')
    inp.append('*Element,type=U3,elset=AllEls')
    for eid, (_, IEN_row, _) in enumerate(elements, start=1):
        inp.append(f'{eid}, ' + ', '.join(str(k) for k in IEN_row))
    n_elem = len(elements)
    inp += [f'*ELSET,ELSET=EltPatch1,generate', f'1,{n_elem},1',
            f'*ELSET,ELSET=EltToLoad,generate', f'1,{n_elem},1']
    corner_local = [0, nu_cp - 1, nu_cp * (nv_cp - 1), nu_cp * nv_cp - 1]
    for k, local_pos in enumerate(corner_local):
        gid = patch.global_indices[local_pos]
        inp += [f'*NSET,NSET=CPatCorner{k}', str(gid + 1)]
    inp += [f'*UEL PROPERTY, ELSET=EltPatch1, MATERIAL=MAT', f'1, {thickness:.16g}',
            '*End Part', '*Assembly, name=Assembly', '*Instance, name=I1, part=Piece',
            '*End Instance', '*End Assembly', '*MATERIAL,NAME=MAT', '*Elastic',
            f'{E:.16g}, {nu}', '*STEP,extrapolation=NO,NLGEOM=NO', '*Static', '*Boundary']
    for k in range(4):
        inp.append(f'I1.CPatCorner{k}, 1, 3, 0.')
    inp += ['*Dload', f'I1.EltToLoad, {load_direction_code}, {load_magnitude:.16g}', '*End Step']
    with open(f'{basename}.inp', 'w') as fh:
        fh.write('\n'.join(inp) + '\n')


basename = os.path.join(tempfile.gettempdir(), 'squareroof_refined_legacy')
write_legacy_shell_files(patch_refined, basename, E=210e9, nu=0.3, thickness=0.1,
                          load_direction_code='U66', load_magnitude=-500.0)

legacy_model = IGAparametrization(filename=basename)
data, row, col, F_legacy = build_stiffmatrix_legacy(*legacy_model.get_inputs4system_elemStorage())
K_legacy_r = sp.coo_matrix((data, (row, col)), shape=(legacy_model.nb_dof_tot,) * 2,
                           dtype='float64').tocsc()
K_legacy_r = K_legacy_r + K_legacy_r.transpose()

rel_err_K = np.linalg.norm(K_r.toarray() - K_legacy_r.toarray()) / np.linalg.norm(K_r.toarray())
rel_err_F = np.linalg.norm(F_r - F_legacy) / np.linalg.norm(F_r)
print('relative Frobenius error, K (refined mesh):', rel_err_K)
print('relative error, F (refined mesh):', rel_err_F)

d_legacy = np.zeros(K_legacy_r.shape[0])
d_legacy[free_dofs] = spsolve(K_legacy_r.tocsc()[free_dofs, :][:, free_dofs], F_legacy[free_dofs])

rel_err_d = np.linalg.norm(d_future - d_legacy) / np.linalg.norm(d_future)
print('relative error, displacement field:', rel_err_d)
print('max |uz|: future =', np.abs(d_future[2::3]).max(), ' legacy =', np.abs(d_legacy[2::3]).max())

K, F, and the full solved displacement field all agree to machine precision -- the
refine/clamp/load/solve pipeline is validated against legacy end-to-end.

### VTK export and interactive rendering

`write_bezier_patch_vtu` (`02_patch_basics.ipynb`, `09_quarter_ring_nurbs.ipynb`) writes
the patch as high-order VTK Bézier cells (`VTK_BEZIER_QUADRILATERAL`), with any field
defined at the B-spline control points transformed to the Bézier basis the same way. Here
the field is the raw (unscaled) 3-component displacement. `pyvista` (`pv.read` + a
`Plotter`) reads that same file back and renders it directly in this notebook as an
**interactive** widget (`jupyter_backend='trame'`: rotate/zoom/pan with the mouse) -- no
external application needed. The exported `.vtu` file itself is still there afterwards for
full exploration in Paraview too (`Filters > Alphabetical > Warp By Vector`).

Toggling "edges" on this surface (the widget's own toolbar button, independent of the
Python code) does *not* show the 4x4 element grid the way Paraview's "Surface With
Edges" does -- it shows every edge of the flat triangles VTK tessellated each curved
Bézier cell into for rendering, which is a much finer, denser grid than the actual
elements. Paraview must be drawing the *original* cell boundaries as a separate pass,
not literally "every polygon edge of the tessellated surface". `yeti_iga.future.postprocessing.pv_plotting`
now packages the fix (extracting the true element boundaries straight from the cell
connectivity -- see its docstring) into a reusable `add_deformed_shell()` helper, in the
same spirit as `plotting.py`'s 2D matplotlib helpers: it warps, colors, and overlays the
correct element grid on a `pyvista.Plotter` you provide, in one call.

In [ ]:
os.makedirs('output', exist_ok=True)
displacement = d_future.reshape(n_cp_r, 3)
write_bezier_patch_vtu(patch_refined, 'output/squareroof_deformed.vtu',
                        field=displacement, field_name='displacement')
print('Exported -> output/squareroof_deformed.vtu')

In [ ]:
mesh = pv.read('output/squareroof_deformed.vtu')

pl = pv.Plotter(off_screen=True, window_size=[800, 600])
add_deformed_shell(pl, mesh, factor=100.0, component=2)  # exaggerate: mm-scale deflection on a 10 m plate
pl.add_axes()
pl.camera_position = 'iso'
pl.show(jupyter_backend='trame')

## Next steps

- Multipatch C0 shell assemblies: `assemble_shell_stiffness()` / `assemble_shell_mass()` /
  `assemble_shell_surface_load()` reuse `PatchAssembly`'s shared-control-point resolution
  exactly like the solid path (`06_multipatch_stiffness.ipynb`) -- no shell-specific work
  needed there.
- A direction-varying normal-pressure surface load (legacy `U_0` on curved geometry) is a
  natural extension of `integrate_shell_surface_load()`, not needed yet.
- C1 bending-strip coupling for folded/multi-panel shells remains out of scope.